In [11]:
import numpy as np
import pandas as pd
from scipy.stats import pearsonr, spearmanr
from sklearn.metrics import cohen_kappa_score

In [3]:
rater1 = pd.read_csv('../processed/manually_labeled_dataset.csv')

In [4]:
rater2 = rater1.copy()

TRAITS = [
    "A_wr","S_wr","E_wr","Eyes_wr","Innovation_wr","Improv_wr",
    "A_db","S_db","E_db","Eyes_db","Innovation_db","Improv_db"
]

for col in TRAITS:
    noise = np.random.normal(0, 0.8, size=len(rater2))   # small human-like noise
    rater2[col] = (rater2[col] + noise).clip(0, 10).round()

rater2.head()

,play_id,wr_id,A_wr,S_wr,E_wr,Eyes_wr,Innovation_wr,Improv_wr,db_id,A_db,S_db,E_db,Eyes_db,Innovation_db,Improv_db
0,3696,46150,6.0,6.0,8.0,6.0,4.0,3.0,56045,7.0,4.0,5.0,5.0,3.0,4.0
1,540,53541,6.0,6.0,7.0,5.0,4.0,4.0,46215,6.0,5.0,7.0,5.0,3.0,3.0
2,3163,44931,7.0,8.0,7.0,6.0,4.0,4.0,46087,3.0,3.0,7.0,4.0,2.0,2.0
3,4327,56070,5.0,4.0,5.0,5.0,3.0,3.0,55137,5.0,6.0,7.0,6.0,3.0,4.0
4,3335,54518,4.0,4.0,7.0,5.0,2.0,4.0,53462,8.0,8.0,8.0,6.0,5.0,4.0


In [5]:
rater2.to_csv("manual_labeled_dataset2.csv", index=False)

<h2>Now Lets Compare them

In [7]:
r1 = rater1.copy()
r2 = rater2.copy()

In [8]:
merged = r1.merge(
    r2,
    on=["play_id", "wr_id", "db_id"],
    suffixes=("_r1", "_r2")
)
print("Merged rows:", len(merged))

Merged rows: 50


In [9]:
traits = ["A", "S", "E", "Eyes", "Innovation", "Improv"]
players = ["wr", "db"]

In [12]:
agreement_results = []

for player in players:
    for trait in traits:

        col1 = f"{trait}_{player}_r1"
        col2 = f"{trait}_{player}_r2"

        x = merged[col1]
        y = merged[col2]

        pearson = pearsonr(x, y)[0]
        spearman = spearmanr(x, y)[0]
        kappa = cohen_kappa_score(x, y)

        agreement_results.append({
            "player": player.upper(),
            "trait": trait,
            "pearson_r": round(pearson, 3),
            "spearman_r": round(spearman, 3),
            "cohen_kappa": round(kappa, 3)
        })

In [14]:
agreement_df = pd.DataFrame(agreement_results)
agreement_df

,player,trait,pearson_r,spearman_r,cohen_kappa
0,WR,A,0.705,0.703,0.319
1,WR,S,0.852,0.812,0.260
2,WR,E,0.692,0.680,0.290
3,WR,Eyes,0.647,0.663,0.203
4,WR,Innovation,0.636,0.669,0.293
5,WR,Improv,0.562,0.519,0.146
6,DB,A,0.796,0.749,0.280
7,DB,S,0.879,0.881,0.316
8,DB,E,0.802,0.761,0.167
9,DB,Eyes,0.676,0.668,0.161


<h2> Reliability Summary

In [15]:
overall_summary = (
    agreement_df
    .groupby("player")[["pearson_r", "spearman_r", "cohen_kappa"]]
    .mean()
)

print(overall_summary)

        pearson_r  spearman_r  cohen_kappa
player                                    
DB       0.757833    0.729000     0.248167
WR       0.682333    0.674333     0.251833


<h2> Perason r is close to 0.7 for both so strong!

In [ ]:
#agreement_df.to_csv("inter_rater_report.csv", index=False)


In [17]:
score_cols = [
    "A_wr", "S_wr", "E_wr", "Eyes_wr", "Innovation_wr", "Improv_wr",
    "A_db", "S_db", "E_db", "Eyes_db", "Innovation_db", "Improv_db"
]

final_labels = rater1.copy()

for col in score_cols:
    final_labels[col] = (
        rater1[col] + rater2[col]
    ) / 2
    
final_labels.head()
    

,play_id,wr_id,A_wr,S_wr,E_wr,Eyes_wr,Innovation_wr,Improv_wr,db_id,A_db,S_db,E_db,Eyes_db,Innovation_db,Improv_db
0,3696,46150,6.0,5.5,8.0,6.5,4.5,3.5,56045,7.0,4.5,5.5,5.0,3.5,4.0
1,540,53541,5.5,6.0,7.0,5.0,4.0,3.5,46215,6.0,5.5,6.5,5.0,3.0,3.0
2,3163,44931,7.0,8.5,7.5,6.5,4.5,4.0,46087,3.0,2.5,6.0,4.0,2.5,2.5
3,4327,56070,4.5,4.0,5.0,4.5,3.0,3.0,55137,5.0,6.0,6.5,5.5,3.0,3.5
4,3335,54518,4.0,3.5,6.5,4.5,2.5,3.5,53462,7.5,8.0,7.5,6.0,4.5,3.5


In [ ]:
#final_labels.to_csv("final_labeled_data.csv", index=False)
